# Self-Reflecting Agent (Reflexion) with LangGraph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/18-self-reflecting-agent/reflexion_workbook.ipynb)

---

## What is Reflexion?

**Reflexion** (Shinn et al., 2023) is a technique that lets an LLM improve its own output through a critique-revision loop:

1. **Generate** -- produce an initial answer
2. **Critique** -- evaluate the answer, identify gaps, score confidence
3. **Revise** -- if not yet confident, generate an improved answer using the critique
4. **Repeat** -- until confidence is `high` or a max-iteration cap is reached

This workbook implements the full loop using **LangGraph 0.6.x** -- no retrieval, no external tools, just a self-correcting reasoning cycle.

### Graph

```
START
  |
generate       <- draft (first pass) or revise (subsequent passes)
  |
critique       <- LLM scores confidence: low / medium / high
  |
  +- high confidence ──────────────► END
  +- max iterations reached ───────► END
  +- low / medium ─────────────────► generate  (loop back)
```

### Key LangGraph concepts covered

| Concept | Where |
|---|---|
| `TypedDict` state | `ReflexionState` |
| `with_structured_output` | `Confidence` Pydantic grader |
| Conditional loop | `critique → generate` back-edge |
| Stopping condition | `confident == True` or `iterations >= MAX` |
| Streaming updates | Cell 10 |
| Graph visualisation | Cell 7 |

## Cell 1 -- Install dependencies

Only runs on Google Colab. Skip locally if packages are already installed.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    %pip install -q \
        langchain-core==0.3.79 \
        langchain-openai==0.3.33 \
        langgraph==0.6.7 \
        python-dotenv
    print('Dependencies installed.')
else:
    print('Local environment -- skipping install.')

## Cell 2 -- API key

On Colab: enter your key when prompted. Locally: reads from `.env`.

In [ ]:
import os

if 'google.colab' in sys.modules:
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
else:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY is not set!'
print('API key loaded.')

## Cell 3 -- Prompts and LLM setup

Three prompts drive the loop:

- **`answer_prompt`** -- first pass: generate a fresh answer with no prior context
- **`revise_prompt`** -- subsequent passes: improve the previous answer using the critique
- **`critique_prompt`** -- evaluate the current answer and return a structured `Confidence` score

`with_structured_output` forces the model to return a typed `Confidence` object (`score` + `critique` string) -- no string parsing needed.

In [ ]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel


class Confidence(BaseModel):
    """Structured critique and confidence score for an answer."""
    score: Literal['low', 'medium', 'high']
    critique: str


llm = ChatOpenAI(model='gpt-5-nano')
evaluator = llm.with_structured_output(Confidence)

answer_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer the question thoroughly and accurately.'),
    ('human', '{question}'),
])

revise_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'Improve your answer based on the critique below.\n\n'
        'Original answer:\n{answer}\n\nCritique:\n{critique}\n\n'
        'Write a better answer that addresses every point raised.'
    )),
    ('human', '{question}'),
])

critique_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'Evaluate this answer. Identify factual errors, missing information, '
        'or unclear explanations. '
        'Then rate your confidence that the answer is now correct and complete: '
        'low / medium / high.'
    )),
    ('human', 'Question: {question}\n\nAnswer: {answer}'),
])

print('LLM, evaluator, and prompts ready.')

## Cell 4 -- Smoke test the confidence grader

Before building the graph, verify `with_structured_output` returns a proper `Confidence` object on both a good and a bad answer.

In [ ]:
test_q = 'What is LangGraph?'

test_cases = [
    ('Correct answer',   'LangGraph is a Python library by LangChain for building stateful, multi-actor LLM applications using a graph of nodes and edges.'),
    ('Incorrect answer', 'LangGraph is a graph plotting library for data science and machine learning visualisations.'),
]

for label, ans in test_cases:
    result = (critique_prompt | evaluator).invoke({'question': test_q, 'answer': ans})
    print(f'{label}:')
    print(f'  score    = {result.score!r}')
    print(f'  critique = {result.critique[:140]}')
    print()

## Cell 5 -- Define the Reflexion state

Every node in the graph reads from and writes to `ReflexionState`.

| Field | Type | Purpose |
|---|---|---|
| `question` | `str` | Original question -- never mutated |
| `answer` | `str` | Current best answer; replaced each iteration |
| `critique` | `str` | Critique text from the last evaluation pass |
| `iterations` | `int` | Number of `generate` calls completed |
| `confident` | `bool` | Set `True` when the evaluator scores `high` |

In [ ]:
from typing import TypedDict

MAX_ITERATIONS = 3


class ReflexionState(TypedDict):
    question: str
    answer: str
    critique: str
    iterations: int
    confident: bool


print(f'ReflexionState defined. MAX_ITERATIONS = {MAX_ITERATIONS}')

## Cell 6 -- Build and compile the Reflexion graph

The `generate` node handles both first-pass drafting and subsequent revisions by checking whether `state['answer']` is non-empty. The `should_continue` router returns `END` when the loop should stop and `'generate'` when it should iterate.

In [ ]:
from langgraph.graph import END, START, StateGraph


def generate(state: ReflexionState) -> dict:
    if state['answer']:
        response = (revise_prompt | llm).invoke({
            'question': state['question'],
            'answer':   state['answer'],
            'critique': state['critique'],
        })
    else:
        response = (answer_prompt | llm).invoke({'question': state['question']})
    return {'answer': response.content, 'iterations': state['iterations'] + 1}


def critique(state: ReflexionState) -> dict:
    result = (critique_prompt | evaluator).invoke({
        'question': state['question'],
        'answer':   state['answer'],
    })
    return {'critique': result.critique, 'confident': result.score == 'high'}


def should_continue(state: ReflexionState) -> Literal['generate', '__end__']:
    if state['confident'] or state['iterations'] >= MAX_ITERATIONS:
        return END
    return 'generate'


graph = StateGraph(ReflexionState)
graph.add_node('generate', generate)
graph.add_node('critique', critique)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'critique')
graph.add_conditional_edges('critique', should_continue)

app = graph.compile()
print('Reflexion graph compiled.')

## Cell 7 -- Visualise the graph

The loop back-edge from `critique` to `generate` should be clearly visible.

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'Graph render unavailable: {e}')
    print(app.get_graph().draw_mermaid())

## Cell 8 -- Helper and sample questions

In [ ]:
INIT = lambda q: {
    'question': q, 'answer': '', 'critique': '',
    'iterations': 0, 'confident': False,
}

QUESTIONS = {
    'easy': [
        'What are the main advantages of LangGraph over vanilla LangChain for building agents?',
        'Explain the difference between retrieval and generation in a RAG pipeline.',
        'What is the Reflexion technique and how does it improve LLM reasoning?',
        'How does human-in-the-loop checkpointing work in LangGraph?',
    ],
    'hard': [
        'What is the capital of the moon?',
        'How does LangGraph compare to CrewAI when running more than 100 agents concurrently?',
    ],
}

print(f'{len(QUESTIONS["easy"])} easy questions, {len(QUESTIONS["hard"])} hard questions loaded.')

## Cell 9 -- Run easy questions

These have clear answers. Expect `confident = True` in 1-2 iterations.

In [ ]:
for q in QUESTIONS['easy']:
    r = app.invoke(INIT(q))
    short = r['answer'][:280] + '...' if len(r['answer']) > 280 else r['answer']
    print(f'Q: {q}')
    print(f'   iters={r["iterations"]}  confident={r["confident"]}')
    print(f'   A: {short}')
    print()

## Cell 10 -- Run hard / unanswerable questions

The agent will loop up to `MAX_ITERATIONS` and stop with `confident = False`, demonstrating the safety cap.

In [ ]:
for q in QUESTIONS['hard']:
    r = app.invoke(INIT(q))
    short = r['answer'][:280] + '...' if len(r['answer']) > 280 else r['answer']
    print(f'Q: {q}')
    print(f'   iters={r["iterations"]}  confident={r["confident"]}  hit_cap={not r["confident"]}')
    print(f'   A: {short}')
    print()

## Cell 11 -- Stream the loop step-by-step

`stream_mode='updates'` emits one dict per node as it completes. Watch the confidence score rise (or plateau) across iterations.

In [ ]:
question = 'What are the main advantages of LangGraph over vanilla LangChain for building agents?'
print(f'Streaming Reflexion for: "{question}"\n')

for step in app.stream(INIT(question), stream_mode='updates'):
    node = list(step.keys())[0]
    out  = step[node]
    print(f'[{node}]')
    if 'iterations' in out:
        print(f'  iteration #{out["iterations"]}')
    if 'answer' in out:
        preview = out['answer'][:200]
        print(f'  answer: {preview}{"..." if len(out["answer"]) > 200 else ""}')
    if 'critique' in out:
        print(f'  critique: {out["critique"][:160]}')
    if 'confident' in out:
        print(f'  confident: {out["confident"]}')
    print()

## Exercises

### Exercise 1 -- Tune MAX_ITERATIONS
Change `MAX_ITERATIONS` to `5`. Re-run Cell 9. Do answers meaningfully improve beyond iteration 3, or does the model converge earlier? What does this suggest about diminishing returns?

### Exercise 2 -- Track score history
Add a `score_history: list` field to `ReflexionState`. Update the `critique` node to append the confidence score each round. After the loop, print the progression, e.g. `['low', 'medium', 'high']`.

### Exercise 3 -- Stricter critique
Rewrite `critique_prompt` to require the model to list specific factual claims it cannot verify. Observe how this changes revision behaviour and iteration count.

### Exercise 4 -- Add a checkpointer
Recompile the graph with an in-memory checkpointer:
```python
from langgraph.checkpoint.memory import MemorySaver
app = graph.compile(checkpointer=MemorySaver())
```
Invoke with `config={'configurable': {'thread_id': 'reflexion-demo'}}`. After the run, inspect the full state with `app.get_state(config)`.

### Exercise 5 -- Ground with retrieval (advanced)
Extend the `generate` node to first retrieve documents from a Chroma vectorstore (reuse the setup from `17-corrective-rag`). Update `critique_prompt` to check whether the answer is supported by the retrieved context. How does grounded Reflexion compare?

---

## What's next?

- **Example 25 -- Adaptive RAG** -- Route queries across no-retrieval / vectorstore / web-search paths
- **Example 26 -- RAG Fusion** -- Multi-query + Reciprocal Rank Fusion for higher recall
- **Example 28 -- Parallel Subgraphs** -- LangGraph `Send()` API for true fan-out / fan-in
- **Reflexion paper** -- Shinn et al. (2023), arXiv:2303.11366
- **LangGraph docs** -- https://langchain-ai.github.io/langgraph/